# Weekly Sector Rotation — raw sequential sector state (Track A)

The engineered-feature notebooks (`feature_diagnostics.ipynb`, `neural_network_model.ipynb`)
compress each sector's trailing history into summary statistics (4w/12w return, volatility,
MA distance, drawdown, ...) — a real, lossy compression. The paper this strategy is trying
to replicate may instead have fed something much closer to the **raw N-week price/volume
state of all 11 sectors jointly**:

$$X_t = \begin{bmatrix} P_{1,t-N+1} & P_{2,t-N+1} & \cdots & P_{11,t-N+1} \\
\vdots & \vdots & & \vdots \\ P_{1,t} & P_{2,t} & \cdots & P_{11,t} \end{bmatrix}$$

with every non-target input standardized to zero mean / unit variance. This notebook tests
that hypothesis directly, rather than continuing to tune architecture on the engineered
features (paused per direction from the last session).

## Experiments (A1-A6)

The paper doesn't specify the exact channels, so each is defined precisely here rather than
left ambiguous:

| Exp | Channels (per sector, per week) | Definition |
| --- | --- | --- |
| A1 | price level (window-rebased) | `P_{s,t-i} / P_{s,t-N+1}` — rebased so every window starts at 1.0; removes cross-sector/cross-era absolute price-scale differences without needing global statistics. |
| A2 | price level (training-standardized) | `z_{s,t-i} = (P_{s,t-i} - mu_train_s) / sigma_train_s` — the paper's own stated scheme, using **each sector's own training-fold mean/std only** (fit on training years, applied unchanged to val/test — never refit on data the fold shouldn't see yet). This is the "especially important experiment" from the last discussion. |
| A3 | 1-week return | `P_{s,t-i}/P_{s,t-i-1} - 1` |
| A4 | 1-week return relative to SPY | A3's sector return minus SPY's own 1-week return, same week |
| A5 | A1's price channel + a volume channel | volume channel: `V_{s,t-i} / mean(V_{s,t-N+1..t})` — window-relative, same rebasing logic as A1 |
| A6 | A3 + A4 + A5's volume channel | 3 channels: return, relative return, relative volume |

## Sequence-native architecture

The last session's direction: build an actual sequence model (not a flattened vector into
an MLP) — a single-layer GRU processing the N-week window step by step, hidden size 16,
dropout before a linear head to a single logit. At each timestep the GRU sees a
`11 x channels` vector (all 11 sectors' state that week); its final hidden state (after
seeing all N weeks) feeds the classification head. Kept small deliberately, matching the
"small architecture" reasoning already established for the engineered-feature MLPs — this
is a sequence encoder, not a large model.

**Target-sector-first ordering**: a sample built for predicting sector *s*'s outcome always
places sector *s*'s own channel(s) first among the 11 sectors in the input, with the other
10 in a fixed alphabetical order after it. This lets one shared model serve every sector
(same weight-sharing the engineered-feature models had for free, since here the raw input
would otherwise be permutation-ambiguous about which of the 11 columns is "the" target).

## Staged plan (per the last discussion)

**Pilot** (this notebook, first pass): all 6 experiments at one fixed depth, `N = 13`
(~1 quarter) — cheap enough to run all 6 and see which representation(s), if any, are worth
sweeping further. The full `N in {4, 8, 13, 26, 52}` sweep is deferred to a second pass on
whichever representation(s) look promising here, not run blind across all 30 combinations.

**Evaluation**: same walk-forward-by-year split, same label (`next_week_excess_return > 0`),
same pooled accuracy/AUC/top-K spread metrics as the engineered-feature notebooks, so results
are directly comparable to the standing benchmark (logistic regression and MLP best AUC
~0.50-0.51, no statistically significant top-K spread).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, roc_auc_score

REPO_ROOT = Path.cwd().resolve().parents[3]
PRICES_ROOT = REPO_ROOT / "data" / "raw" / "weekly_sector_rotation" / "prices"
OUTPUTS_ROOT = REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation" / "outputs"

SECTOR_ETFS = ["VGT", "VHT", "VCR", "VOX", "VFH", "VIS", "VDC", "VPU", "VAW", "VNQ", "VDE"]
BENCHMARK_ETF = "SPY"
SYMBOLS = SECTOR_ETFS + [BENCHMARK_ETF]

daily = {}
for symbol in SYMBOLS:
    df = pd.read_csv(PRICES_ROOT / f"{symbol}.csv", parse_dates=["date"])
    daily[symbol] = df.set_index("date").sort_index()


def resample_weekly(df: pd.DataFrame) -> pd.DataFrame:
    weekly = pd.DataFrame(
        {"adjusted_close": df["adjusted_close"].resample("W-FRI").last(), "volume": df["volume"].resample("W-FRI").sum()}
    )
    return weekly.dropna(subset=["adjusted_close"])


weekly = {symbol: resample_weekly(df) for symbol, df in daily.items()}

# Drop the in-progress week's bar -- same rule as feature_selection.ipynb.
last_trading_date = max(df.index.max() for df in daily.values())
current_week_end = last_trading_date + pd.Timedelta(days=(4 - last_trading_date.weekday()) % 7)
if last_trading_date < current_week_end:
    for w in weekly.values():
        w.drop(index=current_week_end, errors="ignore", inplace=True)

adj_close = pd.DataFrame({s: w["adjusted_close"] for s, w in weekly.items()}).sort_index()
volume = pd.DataFrame({s: w["volume"] for s, w in weekly.items()}).sort_index()
adj_close = adj_close.dropna(how="any")  # common-start alignment across all 12 symbols
volume = volume.loc[adj_close.index]

sector_adj_close = adj_close[SECTOR_ETFS]
sector_volume = volume[SECTOR_ETFS]
spy_adj_close = adj_close[BENCHMARK_ETF]

sector_return = sector_adj_close.pct_change()
spy_return = spy_adj_close.pct_change()
sector_relative_return = sector_return.sub(spy_return, axis=0)

print(f"{len(adj_close)} common weekly bars, {adj_close.index.min().date()} -> {adj_close.index.max().date()}")
sector_adj_close.tail()

## Label and window-building utilities

`label_df[t, s]` = sector `s`'s realized relative return the week *after* `t` (same
leak-free construction as every other notebook: shift `sector_relative_return` back one
week within each column). Window building is fully vectorized: stack channels into a
`(n_dates, 11, C)` array, slice every valid `N`-length window once, then expand each
window into 11 samples (one per target sector) via a fixed permutation matrix that moves
each sector into "position 0" in turn — this is what makes one shared GRU able to serve
every sector without an explicit sector-identity feature.

In [ ]:
label_df = sector_relative_return.shift(-1)  # label_df.loc[t, s] = sector s's realized return the week after t

PERM = np.array([[s] + [j for j in range(11) if j != s] for s in range(11)])  # (11, 11): row s = s first, rest alpha order


def build_price_windows(channel_arrays: list[np.ndarray], N: int) -> tuple[np.ndarray, np.ndarray]:
    """channel_arrays: list of (n_dates, 11) arrays, columns in SECTOR_ETFS order.
    Returns (windows: (n_windows, N, 11, C), window_end_positions: (n_windows,))."""
    stacked = np.stack(channel_arrays, axis=-1)  # (n_dates, 11, C)
    n_dates = stacked.shape[0]
    n_windows = n_dates - N + 1
    windows = np.empty((n_windows, N, 11, stacked.shape[-1]), dtype=np.float32)
    for i in range(n_windows):
        windows[i] = stacked[i : i + N]
    return windows, np.arange(N - 1, n_dates)


def expand_to_target_sector_samples(windows: np.ndarray, end_positions: np.ndarray,
                                     dates_index: pd.DatetimeIndex) -> tuple[np.ndarray, np.ndarray, list, list]:
    """Each window -> up to 11 samples (one per target sector), target sector reordered to position 0,
    paired with that sector's next-week label. Samples with an undefined label OR any NaN in the
    window itself (the return channels' very first date has no prior week to compute a return from)
    are dropped -- not just the label check."""
    N = windows.shape[1]
    X_list, y_list, date_list, symbol_list = [], [], [], []
    for i, end_pos in enumerate(end_positions):
        end_date = dates_index[end_pos]
        if end_date not in label_df.index:
            continue
        window = windows[i]
        if np.isnan(window).any():
            continue
        for s_idx, symbol in enumerate(SECTOR_ETFS):
            label = label_df.loc[end_date, symbol]
            if pd.isna(label):
                continue
            reordered = window[:, PERM[s_idx], :].reshape(N, -1)
            X_list.append(reordered)
            y_list.append(label)
            date_list.append(end_date)
            symbol_list.append(symbol)
    X = np.stack(X_list).astype(np.float32)
    y = np.array(y_list, dtype=np.float32)
    return X, y, date_list, symbol_list


print(f"{len(label_df.dropna(how='all'))} dates with at least one defined label")

## Experiment channel definitions

A1, A3, A4, A5, A6 are self-normalizing (window-relative rebasing, or already-scaled
returns) and can be built **once**, outside the walk-forward loop. A2 is the one
exception — its normalization depends on that fold's own training-year statistics, so
it's rebuilt fresh inside each fold instead (see the walk-forward function below).

In [ ]:
raw_price = sector_adj_close.to_numpy()
raw_volume = sector_volume.to_numpy()
raw_return = sector_return.to_numpy()
raw_relative_return = sector_relative_return.to_numpy()
DATES_INDEX = sector_adj_close.index


def rebase_window_channel(channel: np.ndarray) -> np.ndarray:
    """(n_windows, N, 11) -> each window divided by its own first row (window starts at 1.0)."""
    return channel / channel[:, 0:1, :]


def relative_volume_window_channel(channel: np.ndarray) -> np.ndarray:
    """(n_windows, N, 11) -> each window divided by its own mean (window-relative volume)."""
    return channel / channel.mean(axis=1, keepdims=True)


def build_experiment_windows(experiment: str, N: int, price_override: np.ndarray | None = None) -> tuple[np.ndarray, np.ndarray]:
    """Returns (windows: (n_windows, N, 11, C), end_positions). price_override supplies an
    already-standardized price array for A2 (fold-specific training stats); ignored otherwise."""
    price = price_override if price_override is not None else raw_price

    if experiment == "A1":
        windows, end_positions = build_price_windows([price], N)
        windows[..., 0] = rebase_window_channel(windows[..., 0])
    elif experiment == "A2":
        windows, end_positions = build_price_windows([price], N)  # already standardized by caller
    elif experiment == "A3":
        windows, end_positions = build_price_windows([raw_return], N)
    elif experiment == "A4":
        windows, end_positions = build_price_windows([raw_relative_return], N)
    elif experiment == "A5":
        windows, end_positions = build_price_windows([price, raw_volume], N)
        windows[..., 0] = rebase_window_channel(windows[..., 0])
        windows[..., 1] = relative_volume_window_channel(windows[..., 1])
    elif experiment == "A6":
        windows, end_positions = build_price_windows([raw_return, raw_relative_return, raw_volume], N)
        windows[..., 2] = relative_volume_window_channel(windows[..., 2])
    else:
        raise ValueError(experiment)

    return windows, end_positions


EXPERIMENTS = ["A1", "A2", "A3", "A4", "A5", "A6"]
PILOT_N = 13

# Sanity check: build each experiment once at the pilot N and report shapes.
for exp in EXPERIMENTS:
    w, ep = build_experiment_windows(exp, PILOT_N)
    print(f"{exp}: windows {w.shape}, channels={w.shape[-1]}")

## GRU model

In [ ]:
GRU_HIDDEN = 16
GRU_DROPOUT = 0.3
WEIGHT_DECAY = 1e-4
LEARNING_RATE = 1e-3
MAX_EPOCHS = 100
PATIENCE = 8
SEQ_SEED = 20260812
MIN_TRAIN_YEARS = 3
RANK_K_VALUES = [2, 3]


class SectorGRU(nn.Module):
    """Single-layer GRU over the N-week window; final hidden state -> dropout -> linear logit."""

    def __init__(self, input_size: int, hidden_size: int = GRU_HIDDEN, dropout: float = GRU_DROPOUT):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # x: (batch, N, input_size)
        _, h_n = self.gru(x)  # h_n: (1, batch, hidden_size)
        return self.head(self.dropout(h_n.squeeze(0))).squeeze(-1)


def train_fold_gru(X_fit: np.ndarray, y_fit_binary: np.ndarray, X_val: np.ndarray, y_val_binary: np.ndarray,
                    input_size: int, seed: int = SEQ_SEED) -> SectorGRU:
    torch.manual_seed(seed)
    model = SectorGRU(input_size)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss()

    X_fit_t = torch.tensor(X_fit, dtype=torch.float32)
    y_fit_t = torch.tensor(y_fit_binary, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val_binary, dtype=torch.float32)

    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for _ in range(MAX_EPOCHS):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(model(X_fit_t), y_fit_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val_t), y_val_t).item()

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    return model

## Walk-forward evaluation

Fold test years are derived from the dates actually available at this `N` (the panel's
own date range minus the `N-1`-week warmup), not from a fixed constant — a deeper `N`
loses more early history to warmup, so the exact set of usable years shifts with it.
`A2` rebuilds its (fold-specific, training-years-only) standardized dataset inside the
loop; every other experiment reuses one precomputed dataset across all folds, subsetting
by year — architecturally uniform (one function handles both cases) even though it's a
bit wasteful for the non-A2 experiments.

In [ ]:
def top_bottom_spread_from_scores(frame: pd.DataFrame, score_col: str, k: int) -> pd.Series:
    ranked = frame[score_col].rank(ascending=False, method="first")
    n = len(frame)
    top_return = frame.loc[ranked <= k, "next_week_excess_return"].mean()
    bottom_return = frame.loc[ranked > n - k, "next_week_excess_return"].mean()
    return pd.Series({"top_return": top_return, "bottom_return": bottom_return})


def walk_forward_evaluate_experiment(experiment: str, N: int) -> dict:
    years_at_N = sorted(pd.Index(DATES_INDEX[N - 1 :]).year.unique())
    test_years = years_at_N[MIN_TRAIN_YEARS:]

    fixed_dataset = None
    if experiment != "A2":
        windows, end_positions = build_experiment_windows(experiment, N)
        fixed_dataset = expand_to_target_sector_samples(windows, end_positions, DATES_INDEX)

    oof_records = []
    for test_year in test_years:
        train_years_available = [y for y in years_at_N if y < test_year]
        if len(train_years_available) < 2:
            continue
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]

        if experiment == "A2":
            train_mask_dates = DATES_INDEX.year < test_year
            mu = raw_price[train_mask_dates].mean(axis=0)
            sigma = raw_price[train_mask_dates].std(axis=0)
            standardized_price = (raw_price - mu) / sigma
            windows, end_positions = build_experiment_windows("A2", N, price_override=standardized_price)
            X_full, y_full, dates_full, symbols_full = expand_to_target_sector_samples(windows, end_positions, DATES_INDEX)
        else:
            X_full, y_full, dates_full, symbols_full = fixed_dataset

        years_full = np.array([pd.Timestamp(d).year for d in dates_full])
        dates_arr = np.array(dates_full)
        symbols_arr = np.array(symbols_full)

        fit_mask = np.isin(years_full, fit_years)
        val_mask = years_full == val_year
        test_mask = years_full == test_year
        if fit_mask.sum() == 0 or val_mask.sum() == 0 or test_mask.sum() == 0:
            continue

        y_fit_binary = (y_full[fit_mask] > 0).astype(np.float32)
        y_val_binary = (y_full[val_mask] > 0).astype(np.float32)

        model = train_fold_gru(X_full[fit_mask], y_fit_binary, X_full[val_mask], y_val_binary,
                                input_size=X_full.shape[-1])
        with torch.no_grad():
            predicted_proba = torch.sigmoid(model(torch.tensor(X_full[test_mask], dtype=torch.float32))).numpy()

        for d, sym, label, proba in zip(dates_arr[test_mask], symbols_arr[test_mask], y_full[test_mask], predicted_proba):
            oof_records.append({"date": d, "symbol": sym, "next_week_excess_return": float(label),
                                 "label_binary": int(label > 0), "predicted_proba": float(proba)})

    oof = pd.DataFrame(oof_records)
    accuracy = accuracy_score(oof["label_binary"], oof["predicted_proba"] >= 0.5)
    auc = roc_auc_score(oof["label_binary"], oof["predicted_proba"])

    spread_stats = {}
    for k in RANK_K_VALUES:
        weekly = oof.groupby("date").apply(top_bottom_spread_from_scores, score_col="predicted_proba", k=k)
        spread = (weekly["top_return"] - weekly["bottom_return"]).dropna()
        spread_stats[f"mean_spread_k{k}"] = spread.mean()
        spread_stats[f"hit_rate_k{k}"] = (spread > 0).mean()

    return {"accuracy": accuracy, "auc": auc, "n_test_rows": len(oof), "n_test_years": len(test_years), **spread_stats}

## Run the pilot: A1-A6 at N=13

In [ ]:
pilot_rows = []
for exp in EXPERIMENTS:
    metrics = walk_forward_evaluate_experiment(exp, PILOT_N)
    pilot_rows.append({"experiment": exp, "N": PILOT_N, **metrics})
    print(f"{exp}: accuracy={metrics['accuracy']:.4f}, auc={metrics['auc']:.4f}, "
          f"k3_spread={metrics['mean_spread_k3']:.5f}, n_test_rows={metrics['n_test_rows']}")

pilot_results = pd.DataFrame(pilot_rows)
pilot_results.to_csv(OUTPUTS_ROOT / "raw_sequence_pilot_results.csv", index=False)
pilot_results

## Block-bootstrap check on A5 (best pilot result)

Same method as `feature_diagnostics.ipynb` Stage 8 and `neural_network_model.ipynb`
(moving block bootstrap, `BLOCK_SIZE=8` weeks, `N_BOOTSTRAP=5000`, 95% empirical CI) —
A5 has the best AUC of anything built in this project so far, so it gets the same
significance check before being read as a real result rather than noise.

In [ ]:
def walk_forward_oof_experiment(experiment: str, N: int) -> pd.DataFrame:
    """Same procedure as walk_forward_evaluate_experiment, returning the raw out-of-fold
    predictions instead of aggregated metrics -- needed for the bootstrap."""
    years_at_N = sorted(pd.Index(DATES_INDEX[N - 1 :]).year.unique())
    test_years = years_at_N[MIN_TRAIN_YEARS:]

    fixed_dataset = None
    if experiment != "A2":
        windows, end_positions = build_experiment_windows(experiment, N)
        fixed_dataset = expand_to_target_sector_samples(windows, end_positions, DATES_INDEX)

    oof_records = []
    for test_year in test_years:
        train_years_available = [y for y in years_at_N if y < test_year]
        if len(train_years_available) < 2:
            continue
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]

        if experiment == "A2":
            train_mask_dates = DATES_INDEX.year < test_year
            mu = raw_price[train_mask_dates].mean(axis=0)
            sigma = raw_price[train_mask_dates].std(axis=0)
            standardized_price = (raw_price - mu) / sigma
            windows, end_positions = build_experiment_windows("A2", N, price_override=standardized_price)
            X_full, y_full, dates_full, symbols_full = expand_to_target_sector_samples(windows, end_positions, DATES_INDEX)
        else:
            X_full, y_full, dates_full, symbols_full = fixed_dataset

        years_full = np.array([pd.Timestamp(d).year for d in dates_full])
        dates_arr = np.array(dates_full)
        symbols_arr = np.array(symbols_full)

        fit_mask = np.isin(years_full, fit_years)
        val_mask = years_full == val_year
        test_mask = years_full == test_year
        if fit_mask.sum() == 0 or val_mask.sum() == 0 or test_mask.sum() == 0:
            continue

        y_fit_binary = (y_full[fit_mask] > 0).astype(np.float32)
        y_val_binary = (y_full[val_mask] > 0).astype(np.float32)

        model = train_fold_gru(X_full[fit_mask], y_fit_binary, X_full[val_mask], y_val_binary,
                                input_size=X_full.shape[-1])
        with torch.no_grad():
            predicted_proba = torch.sigmoid(model(torch.tensor(X_full[test_mask], dtype=torch.float32))).numpy()

        for d, sym, label, proba in zip(dates_arr[test_mask], symbols_arr[test_mask], y_full[test_mask], predicted_proba):
            oof_records.append({"date": d, "symbol": sym, "next_week_excess_return": float(label),
                                 "predicted_proba": float(proba)})

    return pd.DataFrame(oof_records)


def block_bootstrap_ci(values: pd.Series, block_size: int = 8, n_boot: int = 5000,
                        ci: float = 0.95, seed: int = 20260812) -> dict:
    arr = values.to_numpy()
    n = len(arr)
    rng = np.random.default_rng(seed)
    n_blocks_needed = int(np.ceil(n / block_size))
    max_start = n - block_size

    boot_means = np.empty(n_boot)
    for b in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=n_blocks_needed)
        sample = np.concatenate([arr[s : s + block_size] for s in starts])[:n]
        boot_means[b] = sample.mean()

    lower, upper = np.percentile(boot_means, [(1 - ci) / 2 * 100, (1 + ci) / 2 * 100])
    return {"point_estimate": arr.mean(), "ci_lower": lower, "ci_upper": upper, "n_weeks": n,
            "significant": bool(lower > 0 or upper < 0)}


a5_oof = walk_forward_oof_experiment("A5", PILOT_N)

a5_bootstrap_rows = []
for k in RANK_K_VALUES:
    weekly = a5_oof.groupby("date").apply(top_bottom_spread_from_scores, score_col="predicted_proba", k=k)
    spread = (weekly["top_return"] - weekly["bottom_return"]).dropna()
    result = block_bootstrap_ci(spread)
    a5_bootstrap_rows.append({"target": f"A5_N{PILOT_N}_k{k}", **result})

a5_bootstrap_summary = pd.DataFrame(a5_bootstrap_rows)
a5_bootstrap_summary.to_csv(OUTPUTS_ROOT / "raw_sequence_a5_block_bootstrap_ci.csv", index=False)
a5_bootstrap_summary

## Findings (pilot, N=13)

**A2 — the paper's own explicitly-stated normalization scheme — performed worst, not
best.** AUC 0.4955 (below chance) and negative spreads at both K. This directly
contradicts the a priori expectation that training-set standardization would matter
most. Possible explanations, none confirmed here: fixed training-fold mean/std doesn't
adapt within a fold (a genuine regime shift partway through a training window would
leave the z-score anchored to a stale mean); or 11 independent per-sector scalars give
the GRU less directly comparable cross-sectional information than A1's window-relative
rebasing does. Worth investigating further before concluding the paper's scheme simply
doesn't work here, but the naive expectation was wrong.

**AUC and top-K spread disagree, and spread is what matters for a portfolio.** A5
(price + volume) has the best AUC of anything built in this project (0.5119), better
than the strongest engineered-feature MLP (0.5100). But its own top-K spread is
essentially zero (K2: +0.00003, K3: +0.00001) — nowhere near the MLP's best spread
(+0.0013) — and the block-bootstrap CI comfortably includes zero at both K
(`[-0.00128, +0.00127]` at K2). A model can rank the *overall* distribution slightly
better (AUC) while barely separating the *extremes* that a top-K portfolio actually
trades on. AUC alone would have been a misleading signal to act on here.

**A1, A3, A4 cluster together** (AUC 0.507-0.509, small positive spreads, not
independently bootstrap-checked here but of similar magnitude to results already shown
insignificant elsewhere in this project) — directionally consistent with each other,
consistent with the weak-signal picture from every prior notebook, no reason to expect
significance either.

**A6 is middling** (AUC 0.5025, negative spreads) — combining return + relative return +
volume didn't help over the simpler individual representations.

## Recommendation for the N-sweep (second pass)

Per the staged plan: sweep `N in {4, 8, 26, 52}` (13 already done) on **A5** first (best
AUC) and **A1** (simplest, second-best AUC, cheapest to interpret) — not all 6. Skip A2
unless the training-standardization question specifically warrants its own follow-up,
skip A6 (no better than its components), and treat A3/A4 as adequately represented by
A1's similar performance rather than independently sweeping every representation.
Not run in this pass — deferred pending direction, consistent with staging this as a
pilot rather than committing to the full 30-combination sweep blind.

## N-sweep: A5 and A1 at N in {4, 8, 26, 52}

The remaining depths from the original `N in {4, 8, 13, 26, 52}` sweep, on the two
representations recommended above (N=13 already run for both, reused from the pilot
rather than rerun). A deeper `N` costs more per fold (more GRU timesteps to
backpropagate through) and loses more early history to warmup — both effects show up
directly in the runtime and `n_test_years`/`n_test_rows` columns below.

In [ ]:
SWEEP_N_VALUES = [4, 8, 26, 52]
SWEEP_EXPERIMENTS = ["A5", "A1"]

sweep_rows = list(pilot_rows)  # include the already-run N=13 results for A1/A5 (and the rest, for completeness)
for exp in SWEEP_EXPERIMENTS:
    for N in SWEEP_N_VALUES:
        metrics = walk_forward_evaluate_experiment(exp, N)
        sweep_rows.append({"experiment": exp, "N": N, **metrics})
        print(f"{exp} N={N}: accuracy={metrics['accuracy']:.4f}, auc={metrics['auc']:.4f}, "
              f"k3_spread={metrics['mean_spread_k3']:.5f}, n_test_rows={metrics['n_test_rows']}, "
              f"n_test_years={metrics['n_test_years']}")

sweep_results = pd.DataFrame(sweep_rows).sort_values(["experiment", "N"])
sweep_results.to_csv(OUTPUTS_ROOT / "raw_sequence_n_sweep_results.csv", index=False)
sweep_results[sweep_results["experiment"].isin(SWEEP_EXPERIMENTS)]

### Block-bootstrap check on the sweep's best spread

Selecting by `|mean_spread_k3|` (the portfolio-relevant metric), not AUC — the pilot's
A5 result already showed the two can disagree, so AUC alone isn't a reliable way to
pick "the best" result to check.

In [ ]:
candidates = sweep_results[sweep_results["experiment"].isin(SWEEP_EXPERIMENTS)].copy()
best_row = candidates.loc[candidates["mean_spread_k3"].abs().idxmax()]
best_exp, best_N = best_row["experiment"], int(best_row["N"])
print(f"best |k3 spread| in the sweep: {best_exp} at N={best_N} "
      f"(spread={best_row['mean_spread_k3']:.5f}, auc={best_row['auc']:.4f})")

best_oof = walk_forward_oof_experiment(best_exp, best_N)

sweep_bootstrap_rows = []
for k in RANK_K_VALUES:
    weekly = best_oof.groupby("date").apply(top_bottom_spread_from_scores, score_col="predicted_proba", k=k)
    spread = (weekly["top_return"] - weekly["bottom_return"]).dropna()
    result = block_bootstrap_ci(spread)
    sweep_bootstrap_rows.append({"target": f"{best_exp}_N{best_N}_k{k}", **result})

sweep_bootstrap_summary = pd.DataFrame(sweep_bootstrap_rows)
sweep_bootstrap_summary.to_csv(OUTPUTS_ROOT / "raw_sequence_sweep_best_block_bootstrap_ci.csv", index=False)
sweep_bootstrap_summary

### Multiple-comparisons correction on the flagged result

`A1, N=52, K=3` came back significant at a nominal 95% CI — but it was selected as the
single most extreme `|k3 spread|` out of **10** combinations tested in this sweep
(2 experiments x 5 N values). Across 10 independent-ish tests, the chance that *at
least one* looks significant at a nominal 95% level purely from noise is
`1 - 0.95^10 ≈ 40%` — far too high to trust the nominal p-value here. Applying a
Bonferroni correction (the conservative, standard fix for exactly this situation):
significance threshold `alpha = 0.05 / 10 = 0.005`, i.e. the CI needs to be a **99.5%**
interval, not 95%, to survive.

In [ ]:
N_TESTS_IN_SWEEP = len(candidates)  # 10: 2 experiments x 5 N values
BONFERRONI_CI = 1 - 0.05 / N_TESTS_IN_SWEEP

corrected_rows = []
for k in RANK_K_VALUES:
    weekly = best_oof.groupby("date").apply(top_bottom_spread_from_scores, score_col="predicted_proba", k=k)
    spread = (weekly["top_return"] - weekly["bottom_return"]).dropna()
    result_95 = block_bootstrap_ci(spread, ci=0.95)
    result_corrected = block_bootstrap_ci(spread, ci=BONFERRONI_CI)
    corrected_rows.append({
        "target": f"{best_exp}_N{best_N}_k{k}",
        "point_estimate": result_95["point_estimate"],
        "ci95_lower": result_95["ci_lower"], "ci95_upper": result_95["ci_upper"], "ci95_significant": result_95["significant"],
        f"ci{BONFERRONI_CI:.3f}_lower": result_corrected["ci_lower"],
        f"ci{BONFERRONI_CI:.3f}_upper": result_corrected["ci_upper"],
        "bonferroni_significant": result_corrected["significant"],
    })

corrected_summary = pd.DataFrame(corrected_rows)
corrected_summary.to_csv(OUTPUTS_ROOT / "raw_sequence_sweep_best_bonferroni_ci.csv", index=False)
corrected_summary

## Findings (N-sweep)

**The one nominally-significant result does not survive correction.** `A1, N=52, K=3`'s
95% CI (`[-0.00203, -0.00012]`, excludes zero) was flagged as a potential real result
above — but after Bonferroni-correcting for the 10-way search that surfaced it, the
required 99.5% CI is `[-0.00244, +0.00030]`, which includes zero. This is exactly the
false-positive-from-multiple-comparisons scenario the correction step exists to catch,
and it caught it: without checking, this would have been reported as "the first
significant result in the project"; with the correction, it's not distinguishable from
what 10 noisy tests would produce by chance alone.

**Across the full sweep, no representation or depth stands out.** AUC ranges 0.5062-0.5119
for A5 and 0.5068-0.5106 for A1 across all 5 N values — a narrow band with no clear
trend as N increases (deeper history neither reliably helps nor hurts). Spreads are
small and inconsistent in sign at every N for both representations, matching the
pattern from every prior notebook in this project.

**Where Track A leaves the overall investigation**: raw sequential state (this
notebook), engineered summary features + logistic regression, and engineered features +
two MLP architectures (`feature_diagnostics.ipynb`, `neural_network_model.ipynb`) have
all now been tried, all walk-forward validated the same way, all landing on the same
conclusion once properly corrected for multiple comparisons — no statistically
significant weekly top-K signal on this 11-sector universe with this feature set. The
one apparent exception (A1 N=52 K=3) was checked specifically because it looked
promising, and did not hold up. This is a substantive, multi-method null result at this
point, not a gap in what's been tried.